# Week 1 · Notebook 2 — Indicators: turning prices into signals

A raw price tells you almost nothing. Is 18.50 high or low? Rising or calm? An
**indicator** is a small calculation over recent prices that answers one such
question with a number. Today you build two, and put them on the chart tool you made
yesterday.

Two functions you build:
1. `sma` — the simple moving average (the trend).
2. `rsi` — the relative strength index (momentum: overbought vs oversold).

## 1. Function — `sma` (simple moving average)

The average of the last `window` prices, recomputed each day. It smooths daily
noise so a trend becomes visible. Because it needs `window` days of history before
it can produce a value, the first `window-1` entries are `NaN`.

**In:** `prices`, `window`. **Out:** an array the same length as `prices`, `NaN`
for the first `window-1` entries.
**Hint:** for each `i`, average `prices[i-window+1 : i+1]`.
**Done when:** the check passes.

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from tradinglab.data_feed import DataFeed

feed = DataFeed.from_dir('data/egx'); 
price = feed.close[:, 0]

def sma(prices, window):
    prices = np.asarray(prices, dtype=float)
    out = np.full_like(prices, np.nan)
    # ---8<--- solution
    for i in range(window-1, len(prices)):
        out[i] = prices[i-window+1:i+1].mean()
    # ---8<--- end
    return out

t = sma(np.array([1.,2,3,4,5]), 3)
assert np.isnan(t[:2]).all() and t[2]==2.0 and t[4]==4.0, 'not right yet'
print('sma correct ✓')

## 2. Function — `rsi` (relative strength index)

RSI measures how hard price has been pushed up vs down recently, on a 0–100 scale.
Above ~70 is often called "overbought", below ~30 "oversold". The recipe: average
the up-moves and the down-moves over a window, form `rs = avg_gain / avg_loss`, then
`100 - 100/(1+rs)`.

**In:** `prices`, `window` (default 14). **Out:** array in [0, 100], `NaN` early.
**Hint:** `delta = np.diff(prices)`; gains are positive deltas, losses the absolute
negative ones.
**Done when:** values stay within [0, 100].

In [ ]:
def rsi(prices, window=14):
    prices = np.asarray(prices, dtype=float)
    out = np.full_like(prices, np.nan)
    delta = np.diff(prices)
    gains = np.where(delta > 0, delta, 0.0)
    losses = np.where(delta < 0, -delta, 0.0)
    # ---8<--- solution
    for i in range(window, len(prices)):
        ag = gains[i-window:i].mean(); al = losses[i-window:i].mean()
        out[i] = 100.0 if al == 0 else 100.0 - 100.0/(1.0 + ag/al)
    # ---8<--- end
    return out

r = rsi(price, 14); valid = r[~np.isnan(r)]
assert (valid >= 0).all() and (valid <= 100).all(), 'RSI must be in [0,100]'
print('rsi correct ✓  (recent RSI:', round(np.nanmean(r[-20:]),1), ')')

## 3. See them on your chart
Reuse the `plot_price` tool you built yesterday. Indicators only mean something when
you can see them against price.

In [ ]:
from tradinglab.charting import plot_price   # your graduated tool
ax = plot_price(feed.dates, price,
                overlays={'SMA20': sma(price, 20), 'SMA50': sma(price, 50)},
                title=feed.symbols[0] + ' with moving averages')
plt.show()
print('Notice how the SMAs lag price and smooth the noise — that lag is the trade-off.')

In [ ]:
# --- RSI belongs in its own panel — different scale than price ---
r = rsi(price, 14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(feed.dates, price, linewidth=1.2)
ax1.set_title(feed.symbols[0] + ' — price')
ax1.grid(alpha=0.3)

ax2.plot(feed.dates, r, color='purple', linewidth=1.0)
ax2.axhline(70, color='red', linestyle='--', linewidth=0.8, label='overbought (70)')
ax2.axhline(30, color='green', linestyle='--', linewidth=0.8, label='oversold (30)')
ax2.set_ylim(0, 100)
ax2.set_title('RSI(14)')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Graduate and reflect
You now have `sma` and `rsi`. Move them into `src/tradinglab/indicators.py` and run
`uv run pytest week1/tests/`.

These aren't just charts — tomorrow they become **signals**. When a short SMA rises
above a long one, that's a trend you can trade. That's the strategy you build next.

## 5. First strategy — SMA crossover, one stock

A simple rule:

- When the fast moving average crosses **above** the slow one, **buy**.
- When the fast moving average crosses **below** the slow one, **sell**.

Applied to one stock first, this turns the indicators into visible trading signals.

In [ ]:
fast_window, slow_window = 20, 50
fast_sma = sma(price, fast_window)
slow_sma = sma(price, slow_window)

# A position changes only when the fast SMA crosses the slow SMA.
valid = ~np.isnan(fast_sma) & ~np.isnan(slow_sma)
position = np.where(valid & (fast_sma > slow_sma), 1, 0)
change = np.diff(position, prepend=0)
buy = change == 1
sell = change == -1

assert np.all(fast_sma[buy] > slow_sma[buy])
assert np.all(fast_sma[sell] <= slow_sma[sell])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(feed.dates, price, color='black', linewidth=1.1, label=feed.symbols[0] + ' close')
ax.plot(feed.dates, fast_sma, color='tab:blue', linewidth=1.0,
        label=f'SMA{fast_window}')
ax.plot(feed.dates, slow_sma, color='tab:orange', linewidth=1.0,
        label=f'SMA{slow_window}')
ax.scatter(feed.dates[buy], price[buy], marker='^', color='green', s=55,
           zorder=3, label='buy')
ax.scatter(feed.dates[sell], price[sell], marker='v', color='red', s=55,
           zorder=3, label='sell')
ax.set_title(feed.symbols[0] + ' — SMA crossover strategy')
ax.set_ylabel('Price')
ax.grid(alpha=0.3)
ax.legend(loc='best')
plt.tight_layout()
plt.show()
print(f'Buy signals: {buy.sum()} | Sell signals: {sell.sum()}')

## 6. Apply the strategy to every company

Start with **EGP 1,000 for each company** and apply the SMA crossover independently
to every EGX stock. Each company is reported separately, then compared with the
real EGX30 index over the same dates.

In [ ]:
from tradinglab.data_feed import load_egx30_returns
from tradinglab.metrics import max_drawdown
import pandas as pd

initial_capital = 1000.0
market_feed = DataFeed.from_dir('data/egx')
market_prices = market_feed.close

# Apply the crossover independently to every company; each stock has its own cash balance.
market_fast = np.column_stack([
    sma(market_prices[:, i], fast_window)
    for i in range(market_feed.n_assets)
])
market_slow = np.column_stack([
    sma(market_prices[:, i], slow_window)
    for i in range(market_feed.n_assets)
])
active = (~np.isnan(market_fast) & ~np.isnan(market_slow) &
          (market_fast > market_slow))

# Signals selected at t earn the return from t to t+1.
company_returns = active[:-1] * market_feed.returns[1:]
company_curves = np.cumprod(1.0 + company_returns, axis=0)
company_final = initial_capital * company_curves[-1]
company_profit = company_final - initial_capital
company_drawdown = np.array([
    max_drawdown(company_returns[:, i])
    for i in range(market_feed.n_assets)
])

# Load the real EGX30 index on the same calendar for the comparison row.
egx30_returns = load_egx30_returns('data/egx30.csv', market_feed.dates)
if egx30_returns is None:
    raise ValueError('The EGX30 file does not overlap the market data dates enough.')
egx30_returns = egx30_returns[1:]
egx30_curve = np.cumprod(1.0 + egx30_returns)
egx30_final = initial_capital * egx30_curve[-1]
egx30_profit = egx30_final - initial_capital
egx30_drawdown = max_drawdown(egx30_returns)

report = pd.DataFrame({
    'company': market_feed.symbols,
    'final_value_egp': company_final,
    'final_profit_egp': company_profit,
    'max_drawdown': company_drawdown,
}).sort_values('final_profit_egp', ascending=False).reset_index(drop=True)

print(f'Each company starts with EGP {initial_capital:,.2f}')
print(f'Universe: {market_feed.n_assets} companies | {market_feed.n_days} trading days')
print(report.to_string(index=False, formatters={
    'final_value_egp': 'EGP {:,.2f}'.format,
    'final_profit_egp': 'EGP {:,.2f}'.format,
    'max_drawdown': '{:.2%}'.format,
}))
print(f'\nEGX30 | final value: EGP {egx30_final:,.2f} | '
      f'profit: EGP {egx30_profit:,.2f} | max drawdown: {egx30_drawdown:.2%}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
company_order = report['company'].to_numpy()
profit_order = report['final_profit_egp'].to_numpy()
drawdown_order = report['max_drawdown'].to_numpy()
ax1.bar(company_order, profit_order, color=np.where(profit_order >= 0, 'tab:blue', 'tab:red'))
ax1.axhline(egx30_profit, color='tab:orange', linestyle='--',
            label=f'EGX30 profit: EGP {egx30_profit:,.0f}')
ax1.set_ylabel('Final profit (EGP)')
ax1.set_title('SMA crossover final profit by company')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax2.bar(company_order, drawdown_order * 100, color='tab:red')
ax2.axhline(egx30_drawdown * 100, color='tab:orange', linestyle='--',
            label=f'EGX30 max drawdown: {egx30_drawdown:.1%}')
ax2.set_ylabel('Maximum drawdown (%)')
ax2.set_title('SMA crossover risk by company')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

assert np.isfinite(company_returns).all()
assert len(report) == market_feed.n_assets